# Added value of liberal OFFs for predicting Wake delta power: 48h morphological

Wake sibling of `incremental_added_value.ipynb`, asking the same question: do
CLAS-exclusive and LLAS-exclusive OFFs carry unique information about delta power beyond
the conservative OFFs? Here during Wake, where OFFs are far rarer.

## Why Wake needs a different model

Wake has ~5% as many OFFs as NREM, and the big, broad BLAS events are extremely sparse
(median ~110 per group, with several groups below 50). At the epoch scale BLAS area is
~98% zeros, near-degenerate as a continuous predictor, and controlling for BLAS becomes
vacuous. LLAS-exclusive (small OFFs) is ~86% of all Wake OFFs.

So the continuous-area model from the NREM notebook does not transfer. The adaptations:

1. Occurrence (extensive-margin) model, primary. For sparse tiers the only estimable
   signal is occurrence, not amount. Encode each disjoint tier as a binary "any OFF this
   epoch" indicator and fit
   `z(delta) ~ z(blas_any) + z(clas_excl_any) + z(llas_excl_any)`. Each coefficient is
   the unique presence effect of that tier holding the others' presence fixed.
2. Collapsed amount model, secondary. `z(delta) ~ z(cons_area) + z(llas_excl_area)` with
   the conservative set `cons = BLAS + CLAS-exclusive`: the amount question for the tier
   that can support it, reported with the OLS and rank sibling.
3. Marginal baselines (Step 5). Each tier's coefficient alone (have value) next to its
   partial coefficient (add value), the have-vs-add contrast.
4. A tier whose support in a group is `< MIN_NONZERO_EPOCHS` is set NaN for that group and excluded from that tier's pool, with k reported; a group
   where a predictor is wholly constant is skipped entirely.
5. Coarser epoch sweep, keeping the fine 4 s / 10 s grid: `[4, 10, 30, 60, 120] s`.

Inference (HAC SEs, DerSimonian-Laird RE pooling) and robustness (rank sibling, first
differences, Wake-vs-NREM comparison) mirror the NREM notebook.

One provisional caveat: "SWA during Wake" is interpretively loaded, since wake delta may
reflect local-sleep or drowsy intrusions, or movement and EMG artifact. The hypnogram
drops scored Artifact/NoData, but wake LFP delta is noisier than NREM. Everything
recorded here is methodological; no biological reading is asserted.


In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
import statsmodels.api as sm
import xarray as xr

from cnpix_local_sleep import files, hyp

In [ ]:
# ---- config ----
CACHE_PARQUET = pathlib.Path(
    "./outputs/static_added_value/cache/offs_direct_48h.parquet"
)
OUTPUT_DIR = pathlib.Path("./outputs/incremental_added_value_wake")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
save_plots = True

STATE = "Wake"
GROUP_COLS = ["subject", "probe", "structure"]

EPOCH_DURATION = 10.0          # s (primary headline)
EPOCH_SWEEP = [4.0, 10.0, 30.0, 60.0, 120.0]
MIN_STATE_FRAC = 0.8           # keep an epoch only if >=80% of its samples are clean STATE
MIN_EPOCHS = 50                # per-group minimum
MIN_NONZERO_EPOCHS = 10        # a tier needs >=this many occurrence epochs in a group to be pooled
HAC_TARGET_S = 300.0           # Newey-West maxlags ~ 5 min of autocorrelation

# Disjoint tiers (category in the cached frame): BLAS; CLAS = CLAS-but-not-BLAS;
# LLAS = LLAS-but-not-CLAS. Occurrence predictors (binary) and amount predictors.
OCC = ["blas_any", "clas_excl_any", "llas_excl_any"]
OCC_LABELS = {"blas_any": "BLAS", "clas_excl_any": "CLAS-exclusive",
              "llas_excl_any": "LLAS-exclusive"}
COLLAPSED = ["cons_area", "llas_excl_area"]
COLLAPSED_LABELS = {"cons_area": "Conservative (CLAS set)",
                    "llas_excl_area": "LLAS-exclusive"}

## Load Wake OFFs (reuses the categorized 48h morphological cache)

In [ ]:
offs = pd.read_parquet(
    CACHE_PARQUET,
    columns=GROUP_COLS + ["start_time", "area", "category", "state"],
)
offs = offs[offs["state"] == STATE].reset_index(drop=True)
print(f"{len(offs):,} {STATE} cortical OFFs across "
      f"{offs.groupby(GROUP_COLS, observed=True).ngroups} groups")
print(offs["category"].value_counts().to_dict())

## Per-group delta loader (state-masked, finite, log10)

State-parameterized and cached per `(group, state)` so the Wake-vs-NREM comparison can
reuse it. The outcome is delta over all clean-state samples in a window, never
restricted to OFF intervals.


In [ ]:
_MASKED = {}


def load_group_delta(subject, probe, structure, state):
    key = (subject, probe, structure, state)
    if key in _MASKED:
        return _MASKED[key]
    da = xr.load_dataarray(
        files.get_structure_bandpower_path(subject, probe, structure, "delta", True, "inst")
    )
    t = da["time"].values
    v = da.values
    hg = hyp.load_statistical_condition_hypnograms(subject, probe)["Full.Conservative"]
    keep = hg.keep_states([state]).covers_time(t) & np.isfinite(v)
    fs = 1.0 / np.median(np.diff(t[:10000]))
    _MASKED[key] = (t[keep], np.log10(v[keep]), fs)
    return _MASKED[key]

## Epoch table builder (per tier: total area + occurrence indicator)

In [ ]:
def build_epoch_table(group_offs, t, logd, fs, epoch_duration, min_state_frac):
    if t.size == 0:
        return pd.DataFrame()
    edges = np.arange(t[0], t[-1] + epoch_duration, epoch_duration)
    n = len(edges) - 1
    if n < 1:
        return pd.DataFrame()

    s_ep = np.clip(np.searchsorted(edges, t, side="right") - 1, 0, n - 1)
    dsum = np.zeros(n)
    dcnt = np.zeros(n)
    np.add.at(dsum, s_ep, logd)
    np.add.at(dcnt, s_ep, 1)
    valid = dcnt >= (min_state_frac * epoch_duration * fs)
    mean_log_delta = np.full(n, np.nan)
    mean_log_delta[valid] = dsum[valid] / dcnt[valid]

    cols = {}
    for tier, prefix in [("BLAS", "blas"), ("CLAS", "clas_excl"), ("LLAS", "llas_excl")]:
        sub = group_offs[group_offs["category"] == tier]
        area = np.zeros(n)
        if len(sub):
            oe = np.searchsorted(edges, sub["start_time"].to_numpy(), side="right") - 1
            inr = (oe >= 0) & (oe < n)
            np.add.at(area, oe[inr], sub["area"].to_numpy()[inr].astype(float))
        cols[f"{prefix}_area"] = area
        cols[f"{prefix}_any"] = (area > 0).astype(float)

    cons_area = cols["blas_area"] + cols["clas_excl_area"]
    cols["cons_area"] = cons_area
    cols["cons_any"] = (cons_area > 0).astype(float)

    out = pd.DataFrame(cols)
    out["mean_log_delta"] = mean_log_delta
    out["epoch_start"] = edges[:-1]
    return out.loc[valid].reset_index(drop=True)

## Helpers (standardize / rank, HAC fit, RE pooling)

In [ ]:
def zscore(s):
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if sd > 0 else s * 0.0


def prep_columns(df, transform):
    out = df.copy()
    for c in out.columns:
        col = out[c]
        if transform == "rank":
            col = pd.Series(scipy.stats.rankdata(col), index=col.index)
        out[c] = zscore(col)
    return out


def semipartial_scales(df, predictors):
    """sqrt(1 - R2_i) for each predictor, where R2_i is from regressing predictor i
    on the remaining predictors. Multiplying a joint coefficient by this factor gives
    the semipartial (part) correlation: with a standardized outcome it equals
    cor(y, resid_i), and its square is predictor i's incremental R2. The factor is a
    function of the design matrix alone, so the HAC standard error scales by exactly
    the same amount -- no bootstrap or delta method is needed. With one predictor the
    factor is 1, so a marginal coefficient is its own semipartial."""
    out = {}
    for p in predictors:
        others = [q for q in predictors if q != p]
        r2 = (sm.OLS(df[p], sm.add_constant(df[others])).fit().rsquared
              if others else 0.0)
        out[p] = np.sqrt(max(0.0, 1.0 - r2))
    return out


def random_effects_meta(effects, variances):
    eff = np.asarray(effects, float)
    v = np.asarray(variances, float)
    w = 1.0 / v
    fe = np.sum(w * eff) / np.sum(w)
    q = np.sum(w * (eff - fe) ** 2)
    k = len(eff)
    c = np.sum(w) - np.sum(w**2) / np.sum(w)
    tau2 = max(0.0, (q - (k - 1)) / c) if c > 0 else 0.0
    wre = 1.0 / (v + tau2)
    pooled = np.sum(wre * eff) / np.sum(wre)
    se = 1.0 / np.sqrt(np.sum(wre))
    i2 = max(0.0, (q - (k - 1)) / q) * 100 if q > 0 else 0.0
    p = 2 * (1 - scipy.stats.norm.cdf(abs(pooled / se)))
    return dict(pooled=pooled, se=se, ci_lo=pooled - 1.96 * se,
               ci_hi=pooled + 1.96 * se, p=p, tau2=tau2, i_squared=i2, k=k)


def fit_model(epoch_df, predictors, epoch_duration, transform="zscore",
              hac_target_s=HAC_TARGET_S, min_nnz=MIN_NONZERO_EPOCHS):
    """HAC-OLS of standardized (or ranked) predictors on delta. Per-predictor
    coefficient is NaN'd when its in-group support (# occurrence epochs) < min_nnz;
    returns None if too few epochs or any predictor is constant (unidentified)."""
    base = epoch_df[predictors + ["mean_log_delta"]].dropna()
    if len(base) < MIN_EPOCHS:
        return None
    nnz = {p: int((base[p] > 0).sum()) for p in predictors}
    df = prep_columns(base, transform)
    if (df[predictors].std(ddof=0) == 0).any():
        return None
    y = df["mean_log_delta"]
    X = sm.add_constant(df[predictors])
    maxlags = max(1, int(np.ceil(hac_target_s / epoch_duration)))
    fit = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    out = {"n_epochs": len(df), "maxlags": maxlags}
    for p in predictors:
        ok = nnz[p] >= min_nnz
        out[f"beta_{p}"] = fit.params[p] if ok else np.nan
        out[f"se_{p}"] = fit.bse[p] if ok else np.nan
        out[f"p_{p}"] = fit.pvalues[p] if ok else np.nan
        out[f"nnz_{p}"] = nnz[p]
    # Semipartial (part) coefficients -- the reported partial quantity. beta_sr**2 is
    # the tier's incremental R2; beta**2 (the joint coefficient) is not a variance share.
    scales = semipartial_scales(df, predictors)
    for p in predictors:
        out[f"sr_scale_{p}"] = scales[p]
        out[f"beta_{p}_sr"] = out[f"beta_{p}"] * scales[p]
        out[f"se_{p}_sr"] = out[f"se_{p}"] * scales[p]
    return out


def run_model(offs_df, state, predictors, epoch_duration, transform="zscore"):
    recs = []
    for keys, grp in offs_df.groupby(GROUP_COLS, observed=True):
        t, logd, fs = load_group_delta(*keys, state)
        et = build_epoch_table(grp, t, logd, fs, epoch_duration, MIN_STATE_FRAC)
        res = fit_model(et, predictors, epoch_duration, transform)
        if res is not None:
            res.update(dict(zip(GROUP_COLS, keys)))
            recs.append(res)
    return pd.DataFrame(recs)


def pool_predictors(group_df, predictors, labels, suffix=""):
    """suffix="_sr" pools the semipartial coefficients instead of the joint ones."""
    rows = []
    for p in predictors:
        b, s = f"beta_{p}{suffix}", f"se_{p}{suffix}"
        d = group_df.dropna(subset=[b, s])
        if len(d) < 2:
            continue
        m = random_effects_meta(d[b], d[s] ** 2)
        rows.append(dict(tier=labels[p], pooled_std_beta=m["pooled"], ci_lo=m["ci_lo"],
                         ci_hi=m["ci_hi"], p=m["p"], i_squared=m["i_squared"], k=m["k"]))
    return pd.DataFrame(rows)


def fmt_tbl(df, cap):
    return df.style.format({
        "pooled_std_beta": "{:+.4f}", "ci_lo": "{:+.4f}", "ci_hi": "{:+.4f}",
        "p": "{:.2e}", "i_squared": "{:.0f}%",
    }).set_caption(cap)

## Inventory: measure the sparsity before modeling

Per tier and epoch length: mean (across groups) fraction of epochs containing the tier,
mean occurrence-epoch count, and how many of the 29 groups clear `MIN_NONZERO_EPOCHS`.
This dictates which tiers are estimable at which epoch length.


In [ ]:
inv_rows = []
for ep in EPOCH_SWEEP:
    frac = {p: [] for p in OCC}
    nnz = {p: [] for p in OCC}
    for keys, grp in offs.groupby(GROUP_COLS, observed=True):
        t, logd, fs = load_group_delta(*keys, STATE)
        et = build_epoch_table(grp, t, logd, fs, ep, MIN_STATE_FRAC)
        if len(et) < MIN_EPOCHS:
            continue
        for p in OCC:
            frac[p].append(et[p].mean())
            nnz[p].append(int(et[p].sum()))
    for p in OCC:
        f = np.array(frac[p]); c = np.array(nnz[p])
        inv_rows.append(dict(
            epoch_s=ep, tier=OCC_LABELS[p],
            mean_occ_frac=float(f.mean()),
            median_occ_epochs=float(np.median(c)),
            groups_ge_min=int((c >= MIN_NONZERO_EPOCHS).sum()),
            n_groups=len(c),
        ))
inv_df = pd.DataFrame(inv_rows)
display(
    inv_df.style.format({"mean_occ_frac": "{:.3f}", "median_occ_epochs": "{:.0f}"})
    .set_caption(f"{STATE} occurrence inventory by tier and epoch length")
)

## Primary: occurrence (extensive-margin) model

`z(delta) ~ z(blas_any) + z(clas_excl_any) + z(llas_excl_any)` per group, HAC SEs,
RE-pooled. Each coefficient is the unique presence effect of that tier holding the
others' presence fixed, which is its added value in occurrence terms.


In [ ]:
occ_df = run_model(offs, STATE, OCC, EPOCH_DURATION)
occ_pooled = pool_predictors(occ_df, OCC, OCC_LABELS)
# Semipartial (part) coefficients -- the reported partial quantity.
occ_semipartial_pooled = pool_predictors(occ_df, OCC, OCC_LABELS, suffix="_sr")

# Report dropped support per tier (no silent caps).
for p in OCC:
    kept = occ_df[f"beta_{p}"].notna().sum()
    print(f"{OCC_LABELS[p]:16s}: pooled over {kept}/{len(occ_df)} groups "
          f"(median occ-epochs/group = {occ_df[f'nnz_{p}'].median():.0f})")
print()
for _, r in occ_pooled.iterrows():
    print(f"{r['tier']:16s}: presence beta = {r['pooled_std_beta']:+.4f} "
          f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  p = {r['p']:.2e}  "
          f"I2 = {r['i_squared']:.0f}%  k = {int(r['k'])}")
display(fmt_tbl(occ_pooled, f"{STATE}: pooled occurrence (presence) coefficient by tier"))

In [ ]:
def forest(group_df, pooled_row, beta_col, se_col, title, ax):
    d = group_df.dropna(subset=[beta_col, se_col]).reset_index(drop=True)
    order = np.argsort(d[beta_col].values)
    d = d.iloc[order].reset_index(drop=True)
    labels = [" / ".join(str(d.loc[i, c]) for c in GROUP_COLS) for i in range(len(d))]
    y = np.arange(len(d))[::-1]
    ax.errorbar(d[beta_col], y, xerr=1.96 * d[se_col], fmt="o", color="steelblue",
                ecolor="steelblue", elinewidth=1.0, markersize=3.5, capsize=2)
    yo = -1.5
    ax.fill([pooled_row["ci_lo"], pooled_row["pooled_std_beta"], pooled_row["ci_hi"],
             pooled_row["pooled_std_beta"]], [yo, yo + 0.45, yo, yo - 0.45],
            color="firebrick", alpha=0.75)
    ax.axvline(0, color="grey", ls="--", lw=0.8)
    ax.set_yticks([]); ax.set_title(title, fontsize=8)
    ax.set_xlabel("std. presence coef")
    ax.set_ylim(yo - 1, y[0] + 1 if len(y) else 1)


fig, axes = plt.subplots(1, 3, figsize=(12, 5), sharex=True, constrained_layout=True)
for ax, p in zip(axes, OCC):
    pr = occ_pooled[occ_pooled["tier"] == OCC_LABELS[p]]
    if pr.empty:
        ax.set_title(f"{OCC_LABELS[p]}\n(too few groups)", fontsize=8); ax.axis("off"); continue
    forest(occ_df, pr.iloc[0], f"beta_{p}", f"se_{p}",
           f"{OCC_LABELS[p]}\npooled {pr.iloc[0]['pooled_std_beta']:+.3f} "
           f"[{pr.iloc[0]['ci_lo']:+.3f}, {pr.iloc[0]['ci_hi']:+.3f}]", ax)
fig.suptitle(f"{STATE}: occurrence added value by tier", fontsize=10)
if save_plots:
    fig.savefig(OUTPUT_DIR / "forest_occurrence_by_tier.svg")
plt.show()

## Step 5: marginal vs partial (have value vs add value)

Each tier's coefficient alone (marginal: does its presence predict wake-delta at all?)
next to its partial coefficient from the joint model (add value: beyond the other
tiers). A gap between the two columns makes the have-vs-add distinction explicit.


In [ ]:
marg_rows = []
for p in OCC:
    m_df = run_model(offs, STATE, [p], EPOCH_DURATION)  # single-predictor marginal
    pooled = pool_predictors(m_df, [p], OCC_LABELS)
    if not pooled.empty:
        marg_rows.append(pooled.iloc[0])
marg_pooled = pd.DataFrame(marg_rows)


# Build a tidy marginal-vs-partial comparison.
def _eff(row):
    return f"{row['pooled_std_beta']:+.3f} [{row['ci_lo']:+.3f}, {row['ci_hi']:+.3f}]"
wide = pd.DataFrame({
    "tier": occ_pooled["tier"].values,
    "marginal (have value)": [
        _eff(marg_pooled[marg_pooled["tier"] == t].iloc[0])
        if (marg_pooled["tier"] == t).any() else "n/a"
        for t in occ_pooled["tier"].values
    ],
    "partial (add value)": [_eff(r) for _, r in occ_pooled.iterrows()],
}).set_index("tier")
display(wide.style.set_caption(
    f"{STATE}: marginal vs partial presence coefficient (have value vs add value)"
))

## Secondary: collapsed amount model, with rank sibling

`z(delta) ~ z(cons_area) + z(llas_excl_area)`, conservative set = BLAS + CLAS-exclusive.
The amount question for the tier that can support it, in OLS and rank-transform form.


In [ ]:
def methods_wide(long_df, index_cols):
    d = long_df.copy()
    d["beta [95% CI]"] = d.apply(
        lambda r: f"{r['pooled_std_beta']:+.3f} [{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]",
        axis=1,
    )
    return d.pivot(index=index_cols, columns="method", values="beta [95% CI]")


coll_long = []
coll_group_df = None          # retain the OLS per-group amount coefs for the area forest
area_partial_pooled = None    # OLS amount partial (add value) for the area dumbbell/forest
area_semipartial_pooled = None  # the reported partial quantity (part correlation)
for method, tr in [("OLS", "zscore"), ("rank", "rank")]:
    gd = run_model(offs, STATE, COLLAPSED, EPOCH_DURATION, transform=tr)
    pt = pool_predictors(gd, COLLAPSED, COLLAPSED_LABELS)
    pt.insert(0, "method", method)
    coll_long.append(pt)
    if method == "OLS":
        coll_group_df = gd
        area_partial_pooled = pt.drop(columns="method").reset_index(drop=True)
        area_semipartial_pooled = pool_predictors(
            gd, COLLAPSED, COLLAPSED_LABELS, suffix="_sr")
coll_df = pd.concat(coll_long, ignore_index=True)
for _, r in coll_df.sort_values(["tier", "method"]).iterrows():
    print(f"{r['method']:4s} {r['tier']:26s}: amount beta = {r['pooled_std_beta']:+.4f} "
          f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  p = {r['p']:.2e}  k = {int(r['k'])}")
display(methods_wide(coll_df, ["tier"]).style.set_caption(
    f"{STATE}: collapsed amount model, OLS vs rank-transform"
))

## Step 5b: area model, marginal vs partial (have value vs add value)

The amount analog of Step 5, for the two estimable area predictors (`cons_area` = BLAS +
CLAS-exclusive, and `llas_excl_area`). Each tier's single-predictor OLS coefficient
(marginal, alone) beside its partial coefficient from the joint collapsed model (add
value, holding the other fixed). BLAS has no standalone amount coefficient in Wake,
since it is folded into the conservative set because epoch-scale BLAS area is
near-degenerate, so this figure has two tiers rather than three.


In [ ]:
# Area-model marginals (have value): single-predictor OLS per amount tier, RE-pooled,
# so the area dumbbell can show marginal (alone) vs partial (joint) like the NREM panel.
area_marg_rows = []
for p in COLLAPSED:
    m_df = run_model(offs, STATE, [p], EPOCH_DURATION)  # single-predictor amount marginal
    pooled = pool_predictors(m_df, [p], COLLAPSED_LABELS)
    if not pooled.empty:
        area_marg_rows.append(pooled.iloc[0])
area_marg_pooled = pd.DataFrame(area_marg_rows).reset_index(drop=True)
for _, r in area_marg_pooled.iterrows():
    print(f"marginal {r['tier']:26s}: amount beta = {r['pooled_std_beta']:+.4f} "
          f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  k = {int(r['k'])}")

## Model-light cross-check: delta vs LLAS-exclusive area, split by conservative-OFF presence

Pooled across groups, each group's `llas_excl_area` and `mean_log_delta` z-scored first.
Two strata: epochs with and without any conservative (BLAS + CLAS-excl) OFF. Within
each, mean delta against LLAS-exclusive-area quantile, rank-binned and so inherently
tail-robust.


In [ ]:
parts = []
for keys, grp in offs.groupby(GROUP_COLS, observed=True):
    t, logd, fs = load_group_delta(*keys, STATE)
    et = build_epoch_table(grp, t, logd, fs, EPOCH_DURATION, MIN_STATE_FRAC)
    if len(et) < MIN_EPOCHS:
        continue
    d = et[["llas_excl_area", "clas_excl_area", "cons_any", "mean_log_delta"]].dropna().copy()
    for _c in ["llas_excl_area", "clas_excl_area", "mean_log_delta"]:
        d[_c] = zscore(d[_c])
    parts.append(d)
pooled_epochs = pd.concat(parts, ignore_index=True)

n_q = 5
pooled_epochs["q"] = pooled_epochs.groupby("cons_any", observed=True)["llas_excl_area"].transform(
    lambda s: pd.qcut(s.rank(method="first"), n_q, labels=False)
)
agg = (pooled_epochs.groupby(["cons_any", "q"], observed=True)["mean_log_delta"]
       .agg(["mean", "sem"]).reset_index())

fig, ax = plt.subplots(figsize=(6, 4.2), constrained_layout=True)
for cons, sub in agg.groupby("cons_any", observed=True):
    lbl = "epoch has conservative OFF" if cons == 1 else "no conservative OFF"
    ax.errorbar(sub["q"], sub["mean"], yerr=sub["sem"], marker="o", capsize=2, label=lbl)
ax.set_xlabel("LLAS-exclusive OFF area (within-stratum quantile)")
ax.set_ylabel("mean z(log delta)")
ax.set_title(f"{STATE}: delta vs LLAS-exclusive OFF area, split by conservative-OFF presence",
             fontsize=8)
ax.set_xticks(range(n_q)); ax.legend(fontsize=8)
if save_plots:
    fig.savefig(OUTPUT_DIR / "stratified_added_value_picture.svg")
plt.show()

## Robustness 1: epoch-length sensitivity (occurrence model)

Keeps the fine 4 s / 10 s grid and adds coarser epochs. As epochs coarsen, occurrence
fractions rise (see inventory), so sparse tiers become better estimated.

In [ ]:
sweep_rows = []
for ep in EPOCH_SWEEP:
    gd = run_model(offs, STATE, OCC, ep)
    pt = pool_predictors(gd, OCC, OCC_LABELS)
    pt.insert(0, "epoch_s", ep)
    sweep_rows.append(pt)
sweep_df = pd.concat(sweep_rows, ignore_index=True)
sweep_df["beta [95% CI]"] = sweep_df.apply(
    lambda r: f"{r['pooled_std_beta']:+.3f} [{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}] (k={int(r['k'])})",
    axis=1)
display(
    sweep_df.pivot(index="epoch_s", columns="tier", values="beta [95% CI]")
    .style.set_caption(f"{STATE}: pooled occurrence coefficient by tier and epoch length")
)

## Robustness 1b: epoch-length sensitivity (collapsed amount model)

The sweep above is on the occurrence model. Supplementary Table S2b reports the
collapsed amount model, so sweep that one too, for both the OLS and rank-transform fits,
pooling the semipartial, which is the quantity S2b tabulates. `k` is reported per row
because a tier whose in-group support falls below `MIN_NONZERO_EPOCHS` is dropped from
that tier's pool, and support shrinks as epochs get shorter.


In [ ]:
# Collapsed-model epoch sweep x transform, pooling the SEMIPARTIAL (the quantity
# Table S2b tabulates). fit_model derives maxlags from the epoch length, so the HAC
# window stays ~HAC_TARGET_S across the sweep.
coll_rob_rows = []
for ep in EPOCH_SWEEP:
    for method, tr in [("OLS", "zscore"), ("rank", "rank")]:
        gd = run_model(offs, STATE, COLLAPSED, ep, transform=tr)
        if not len(gd):
            print(f"  skip {ep:.0f}s / {method}: no group fits")
            continue
        pt = pool_predictors(gd, COLLAPSED, COLLAPSED_LABELS, suffix="_sr")
        if pt.empty:
            print(f"  skip {ep:.0f}s / {method}: no tier had >=2 poolable groups")
            continue
        pt.insert(0, "method", method)
        pt.insert(0, "epoch_s", ep)
        coll_rob_rows.append(pt)
coll_rob_df = pd.concat(coll_rob_rows, ignore_index=True)

for _, r in coll_rob_df.iterrows():
    print(f"{r['epoch_s']:5.0f}s {r['method']:4s} {r['tier']:26s}: "
          f"semipartial = {r['pooled_std_beta']:+.4f} "
          f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  p = {r['p']:.2e}  "
          f"I2 = {r['i_squared']:.0f}%  k = {int(r['k'])}")

display(
    methods_wide(coll_rob_df, ["epoch_s", "tier"]).style.set_caption(
        f"{STATE}: collapsed amount model, pooled SEMIPARTIAL coefficient by tier, "
        "epoch length and transform (the quantity reported in Supplementary Table S2b)"
    )
)

## Robustness 2: first differences (occurrence model)

Differencing removes slow shared drift, asking whether changes in tier presence track
changes in wake-delta beyond the others. Short HAC window.


In [ ]:
diff_recs = []
for keys, grp in offs.groupby(GROUP_COLS, observed=True):
    t, logd, fs = load_group_delta(*keys, STATE)
    et = build_epoch_table(grp, t, logd, fs, EPOCH_DURATION, MIN_STATE_FRAC)
    base = et[OCC + ["mean_log_delta"]].dropna()
    if len(base) < MIN_EPOCHS + 1:
        continue
    nnz = {p: int((base[p] > 0).sum()) for p in OCC}
    d = prep_columns(base.diff().dropna(), "zscore")
    if (d[OCC].std(ddof=0) == 0).any():
        continue
    fit = sm.OLS(d["mean_log_delta"], sm.add_constant(d[OCC])).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
    rec = dict(zip(GROUP_COLS, keys))
    for p in OCC:
        ok = nnz[p] >= MIN_NONZERO_EPOCHS
        rec[f"beta_{p}"] = fit.params[p] if ok else np.nan
        rec[f"se_{p}"] = fit.bse[p] if ok else np.nan
    diff_recs.append(rec)
diff_pooled = pool_predictors(pd.DataFrame(diff_recs), OCC, OCC_LABELS)
for _, r in diff_pooled.iterrows():
    print(f"first-diff {r['tier']:16s}: presence beta = {r['pooled_std_beta']:+.4f} "
          f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  p = {r['p']:.2e}  k = {int(r['k'])}")
display(fmt_tbl(diff_pooled, f"{STATE}: first-differenced occurrence coefficient by tier"))

## Wake vs NREM: occurrence model in both states

The occurrence model is the common denominator estimable in both states. Run it on
NREM too (reuses the cached delta) and compare the presence coefficients side by side.

In [ ]:
offs_nrem = pd.read_parquet(
    CACHE_PARQUET, columns=GROUP_COLS + ["start_time", "area", "category", "state"]
)
offs_nrem = offs_nrem[offs_nrem["state"] == "NREM"].reset_index(drop=True)

nrem_occ = pool_predictors(run_model(offs_nrem, "NREM", OCC, EPOCH_DURATION), OCC, OCC_LABELS)

cmp = pd.concat([occ_pooled.assign(state="Wake"), nrem_occ.assign(state="NREM")],
                ignore_index=True)
cmp["beta [95% CI]"] = cmp.apply(
    lambda r: f"{r['pooled_std_beta']:+.3f} [{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]", axis=1)
display(
    cmp.pivot(index="tier", columns="state", values="beta [95% CI]")
    .style.set_caption("Occurrence (presence) added value by tier: Wake vs NREM")
)

# Grouped bar: presence coef by tier and state.
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
tiers = list(OCC_LABELS.values())
x = np.arange(len(tiers))
for j, state in enumerate(["NREM", "Wake"]):
    sub = cmp[cmp["state"] == state].set_index("tier").reindex(tiers)
    err = np.vstack([sub["pooled_std_beta"] - sub["ci_lo"], sub["ci_hi"] - sub["pooled_std_beta"]])
    ax.bar(x + (j - 0.5) * 0.4, sub["pooled_std_beta"], 0.4, yerr=err, capsize=3,
           label=state, color=["#4c72b0", "#dd8452"][j], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(tiers, fontsize=8)
ax.set_ylabel("pooled std. presence coef"); ax.axhline(0, color="grey", ls="--", lw=0.8)
ax.set_title("Occurrence added value by tier: Wake vs NREM"); ax.legend()
if save_plots:
    fig.savefig(OUTPUT_DIR / "wake_vs_nrem_occurrence.svg")
plt.show()

## Interpretation

Reading each tier's presence coefficient:

- CI excludes 0: that tier's occurrence carries unique wake-delta information beyond the
  other tiers, added value in the extensive-margin sense, with the sign giving
  direction.
- CI brackets 0: redundant with the other tiers at this timescale.

Read this in light of the inventory. Because BLAS, and to a lesser extent
CLAS-exclusive, are very sparse in Wake, "LLAS-exclusive adds value over BLAS/CLAS" is
partly trivial, since there is little conservative signal to be redundant with. The more
data-appropriate question is the reverse: do the rare conservative OFFs add anything
over the abundant small ones? The same table answers that through the BLAS and
CLAS-exclusive coefficients, with their `k` and support reported.

Limitations: within-subject clustering, since the two-stage RE treats groups as
exchangeable; HAC across state gaps; the provisional morphological source; and the fact
that the occurrence encoding answers the presence question while the collapsed model
answers the amount question for LLAS-exclusive. The caveat at the top about what wake
delta means also remains open, so treat any biological reading as provisional.


## Export figure data

Write the quantities the figure notebook re-consumes.


In [ ]:
import pathlib

DATA = pathlib.Path("./outputs/added_value_data")
DATA.mkdir(parents=True, exist_ok=True)
# Occurrence (presence) model: primary
occ_pooled.to_parquet(DATA / "wake_partial_pooled.parquet")
occ_semipartial_pooled.to_parquet(DATA / "wake_semipartial_pooled.parquet")
marg_pooled.to_parquet(DATA / "wake_marginal_pooled.parquet")
occ_df[GROUP_COLS + ["beta_blas_any", "se_blas_any", "beta_clas_excl_any",
                     "se_clas_excl_any", "beta_llas_excl_any", "se_llas_excl_any",
                     "beta_blas_any_sr", "se_blas_any_sr", "beta_clas_excl_any_sr",
                     "se_clas_excl_any_sr", "beta_llas_excl_any_sr",
                     "se_llas_excl_any_sr"]].to_parquet(
    DATA / "wake_group_partial.parquet")
pooled_epochs[["llas_excl_area", "clas_excl_area", "cons_any", "mean_log_delta"]].to_parquet(
    DATA / "wake_strat_epochs.parquet")
# Collapsed amount (area) model: secondary; two estimable tiers
# (Conservative = BLAS + CLAS-exclusive, and LLAS-exclusive)
area_partial_pooled.to_parquet(DATA / "wake_area_partial_pooled.parquet")
area_semipartial_pooled.to_parquet(DATA / "wake_area_semipartial_pooled.parquet")
# Collapsed-model robustness grid (epoch sweep x rank sibling, semipartial) -- the
# evidence behind the robustness sentence in the manuscript Methods for Table S2b.
coll_rob_df.to_parquet(DATA / "wake_area_robustness.parquet")
area_marg_pooled.to_parquet(DATA / "wake_area_marginal_pooled.parquet")
coll_group_df[GROUP_COLS + ["beta_cons_area", "se_cons_area",
                            "beta_llas_excl_area", "se_llas_excl_area",
                            "beta_cons_area_sr", "se_cons_area_sr",
                            "beta_llas_excl_area_sr", "se_llas_excl_area_sr"]].to_parquet(
    DATA / "wake_area_group_partial.parquet")
print("exported Wake figure data ->", DATA.resolve())